In [ ]:
from utils import preprocess
from utils import models

df_customers, df_products, df_transactions = preprocess.load_complete_dataset_filtered_number_customers(1000)

customer_ids = df_transactions['customer_id'].unique()[:3].tolist()

predictions, df_merged, summary, kmeans_model = models.predict_cluster(df_transactions, df_customers, df_products, customer_ids=customer_ids, K=8, top_n=12, explore_k=True)


In [ ]:
from utils import models

# Diagnóstico: trazar dónde se vacían las recomendaciones
customer_id = customer_ids[0]

X_final, article_ids, _, df_products_enriched = models.clustering_preprocess(df_customers, df_products, df_transactions)
df_clusters, _ = models.fit_product_clustering(X_final, 8, article_ids)
df_merged, _ = models.inspect_clusters(df_products=df_products_enriched, df_clusters=df_clusters)

id_to_idx = {aid: i for i, aid in enumerate(article_ids)}

bought_raw = df_transactions[df_transactions['customer_id'] == customer_id]['article_id'].unique()
bought = [a for a in bought_raw if a in id_to_idx]

print(f"customer_id: {customer_id}")
print(f"artículos comprados (raw): {len(bought_raw)}")
print(f"artículos comprados en catálogo (bought): {len(bought)}")
print(f"dtype article_id en transactions: {df_transactions['article_id'].dtype}")
print(f"dtype article_id en products:     {df_products['article_id'].dtype}")

if len(bought) > 0:
    clusters_bought = df_merged[df_merged['article_id'].isin(bought)]['cluster'].unique()
    print(f"clusters del cliente: {clusters_bought}")
    candidates = df_merged[(df_merged['cluster'].isin(clusters_bought)) & (~df_merged['article_id'].isin(bought))]
    print(f"candidatos: {len(candidates)}")


# Evaluación: MAP@12 — Cluster vs Baselines

Split temporal: entrenamos con todo excepto la última semana, y evaluamos prediciendo esa última semana.

In [ ]:
import pandas as pd

df_transactions['t_dat'] = pd.to_datetime(df_transactions['t_dat'])

# Leave-one-out: el test es el último artículo comprado por cada usuario
last_purchase_idx = df_transactions.groupby('customer_id')['t_dat'].idxmax()
df_test  = df_transactions.loc[last_purchase_idx]
df_train = df_transactions.drop(index=last_purchase_idx)

# Solo usuarios con historial en train (al menos 1 compra previa)
users_with_train = set(df_train['customer_id'].unique())
eval_users = [u for u in df_test['customer_id'].unique() if u in users_with_train]

ground_truth = df_test.set_index('customer_id')['article_id'].to_dict()
actual = [[ground_truth[u]] for u in eval_users]

print(f"Usuarios para evaluación: {len(eval_users)}")
print(f"Transacciones train: {len(df_train):,} | test: {len(df_test):,}")


In [ ]:
# Diagnóstico del split
items_per_user = [len(v) for v in ground_truth.values()]
print(f"Usuarios en evaluación:              {len(eval_users)}")
print(f"Media de artículos test por usuario: {sum(items_per_user)/len(items_per_user):.2f}")
print(f"Usuarios con ≥1 artículo en test:    {sum(1 for x in items_per_user if x >= 1)}")
print(f"Artículos únicos en catálogo:        {df_products['article_id'].nunique():,}")


In [ ]:
from utils import models

K = 12

# --- Baseline: aleatorio ---
pred_random = models.predict_random(df_train, eval_users, k=K)

# --- Baseline: popularidad ---
pred_popular = models.predict_popular(df_train, eval_users, k=K)

# --- Modelo: clustering ---
pred_cluster, _, _, _ = models.predict_cluster(
    df_train, df_customers, df_products,
    customer_ids=eval_users, K=8, top_n=K, explore_k=False,
)

# --- Evaluación MAP@12 ---
results = {
    'Random':    models.mapk(actual, pred_random,  k=K),
    'Popular':   models.mapk(actual, pred_popular, k=K),
    'Cluster':   models.mapk(actual, pred_cluster, k=K),
}

print("MAP@12:")
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name:<10} {score:.4f}")
